# Temporal Deep Learning for Crop Yield Prediction Using Real Data
**B.Tech Project | Google Colab Ready**

Real crop yield data from FAOSTAT and real environmental data from NASA POWER.

In [ ]:
!pip -q install pandas numpy requests scikit-learn tensorflow matplotlib
import os, zipfile, glob, time, requests, warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
warnings.filterwarnings('ignore')
np.random.seed(42); tf.random.set_seed(42)


## 1. Download official FAOSTAT crop-yield data

In [ ]:
URL='https://bulks-faostat.fao.org/production/Production_Crops_Livestock_E_All_Data_(Normalized).zip'
ZIP='faostat_crops.zip'; OUT='faostat_data'
if not os.path.exists(ZIP):
    r=requests.get(URL,timeout=180); r.raise_for_status()
    open(ZIP,'wb').write(r.content)
os.makedirs(OUT,exist_ok=True)
with zipfile.ZipFile(ZIP) as z: z.extractall(OUT)
files=glob.glob(OUT+'/**/*.csv',recursive=True)
print(files[:5])


In [ ]:
crop_file=next(f for f in files if 'All_Data' in os.path.basename(f))
raw=pd.read_csv(crop_file,encoding='latin-1')
india=raw[raw['Area'].eq('India')].copy()
major=['Wheat','Rice','Maize']
d=india[india['Item'].isin(major) & india['Element'].astype(str).str.lower().str.contains('yield')].copy()
d=d.rename(columns={'Year':'year','Item':'crop','Value':'yield_raw','Unit':'yield_unit'})[['year','crop','yield_raw','yield_unit']]
d['year']=pd.to_numeric(d['year'],errors='coerce')
d['yield_raw']=pd.to_numeric(d['yield_raw'],errors='coerce')
d=d.dropna()
d['yield_tonnes_per_ha']=np.where(d['yield_unit'].astype(str).str.contains('hg/ha',case=False,na=False),d['yield_raw']/10000,d['yield_raw'])
d=d.sort_values(['crop','year']).reset_index(drop=True)
print(d.groupby('crop')['year'].agg(['min','max','count']))


## 2. Download real environmental data from NASA POWER

In [ ]:
# Robust NASA POWER download: request one year at a time to avoid oversized API responses
LAT,LON=22.9734,78.6569
start_year,end_year=int(d['year'].min()),int(d['year'].max())
parameters='T2M,PRECTOTCORR,RH2M,ALLSKY_SFC_SW_DWN'
API='https://power.larc.nasa.gov/api/temporal/daily/point'

def download_power_year(year):
    query={'parameters':parameters,'community':'AG','longitude':LON,'latitude':LAT,'start':f'{year}0101','end':f'{year}1231','format':'JSON'}
    response=requests.get(API,params=query,timeout=180)
    response.raise_for_status()
    payload=response.json()
    parameter_data=payload.get('properties',{}).get('parameter')
    if not parameter_data:
        raise ValueError(f'Unexpected NASA POWER response for {year}: {payload}')
    frame=pd.DataFrame(parameter_data)
    frame.index=pd.to_datetime(frame.index.astype(str),format='%Y%m%d',errors='coerce')
    return frame[frame.index.notna()].replace(-999,np.nan).apply(pd.to_numeric,errors='coerce')

weather_frames=[]; failed_years=[]
for year in range(start_year,end_year+1):
    try:
        weather_frames.append(download_power_year(year))
        print('Downloaded',year)
        time.sleep(0.2)
    except Exception as e:
        print('Warning - failed',year,':',e)
        failed_years.append(year)

if not weather_frames: raise RuntimeError('NASA POWER download failed for all years. Check Colab internet access and rerun.')
w=pd.concat(weather_frames).sort_index()
a=(w.resample('YE').agg({'T2M':'mean','PRECTOTCORR':'sum','RH2M':'mean','ALLSKY_SFC_SW_DWN':'mean'}).reset_index())
a['year']=a['index'].dt.year
a=a.drop(columns='index').dropna(how='all')
a.to_csv('nasa_power_environmental_data.csv',index=False)
print('Shape:',a.shape,'Failed years:',failed_years if failed_years else 'None')
display(a.head())


## 3. Merge real data and create temporal sequences

In [ ]:
df=d[['year','crop','yield_tonnes_per_ha']].merge(a,on='year',how='inner')
df=pd.get_dummies(df,columns=['crop'],dtype=int).sort_values('year').reset_index(drop=True)
df.to_csv('real_crop_environment_dataset.csv',index=False)
features=['T2M','PRECTOTCORR','RH2M','ALLSKY_SFC_SW_DWN']+[x for x in df.columns if x.startswith('crop_')]
WINDOW=5; X=[]; y=[]; yrs=[]
for cc in [x for x in df.columns if x.startswith('crop_')]:
    p=df[df[cc]==1].sort_values('year')
    for i in range(WINDOW,len(p)):
        X.append(p[features].iloc[i-WINDOW:i].values)
        y.append(p['yield_tonnes_per_ha'].iloc[i]); yrs.append(p['year'].iloc[i])
X=np.asarray(X,float); y=np.asarray(y,float); yrs=np.asarray(yrs)
cut=np.quantile(yrs,0.8); tr=yrs<=cut
sc=StandardScaler(); n,steps,nf=X[tr].shape
Xtr=sc.fit_transform(X[tr].reshape(-1,nf)).reshape(n,steps,nf)
Xte=sc.transform(X[~tr].reshape(-1,nf)).reshape(X[~tr].shape)
ytr,yte=y[tr],y[~tr]
print('Train:',Xtr.shape,'Test:',Xte.shape,'Split year:',cut)


## 4. Train LSTM and evaluate

In [ ]:
model=Sequential([LSTM(64,return_sequences=True,input_shape=(WINDOW,len(features))),Dropout(0.2),LSTM(32),Dropout(0.2),Dense(16,activation='relu'),Dense(1)])
model.compile(optimizer='adam',loss='mse',metrics=['mae'])
model.fit(Xtr,ytr,validation_split=0.2,epochs=100,batch_size=16,callbacks=[EarlyStopping(patience=12,restore_best_weights=True)],verbose=1)
pred=model.predict(Xte,verbose=0).ravel()
print('MAE:',mean_absolute_error(yte,pred))
print('RMSE:',mean_squared_error(yte,pred)**0.5)
print('R2:',r2_score(yte,pred))
plt.figure(figsize=(8,4)); plt.plot(yte,marker='o',label='Actual'); plt.plot(pred,marker='x',label='Predicted'); plt.legend(); plt.grid(); plt.title('Actual vs Predicted Crop Yield'); plt.show()
model.save('crop_yield_lstm.keras')
